# 05 Run the detection in batches

Extra lab · 60–90 minutes. Read [the lab guide](guides/05_Run_the_detection_in_batches.md) for the brief, table contracts and acceptance criteria. Work after notebooks 03–04. All evidence is synthetic and local. Edit `labs/scheduled_detection/pipeline.sql`; keep the behavioral rule unchanged.

In [ ]:
from pathlib import Path
import sys
import tempfile

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'labs/scheduled_detection/practice.json').exists()), None)
if root is None:
    raise FileNotFoundError('Open this notebook inside the extracted workshop repository.')
sys.path.insert(0, str(root))
from workshop.scheduled_detection import Replay, SimulatedCrash, rows

# A fresh temporary database per attempt; durable across connection restarts.
attempt = tempfile.TemporaryDirectory(prefix='scheduled_detection-notebook-')
database = Path(attempt.name) / 'state.duckdb'
schedule = root / 'labs/scheduled_detection/practice.json'
pipeline = root / 'labs/scheduled_detection/pipeline.sql'
replay = Replay(database, schedule, pipeline)


## Observe delivery order

Run the next cell repeatedly to advance one batch. Compare event time and ingestion time. Only received evidence is visible to SQL.

In [ ]:
replay.step()
rows(replay.db, """SELECT provider, account_uid, cluster_name, event_id,
    time, ingested_at, operation, resource_type
    FROM deliveries ORDER BY ingested_at DESC LIMIT 12""")

## Reproduce the incident

Finish the replay and inspect the notification outbox. Find a missed sequence and a repeated notification. Explain the difference between event, ingestion and processing time before changing SQL.

In [ ]:
report = replay.run()
report

Inspect repeated notifications by the full asset, grant and recipient key.

In [ ]:
rows(replay.db, """SELECT provider, account_uid, cluster_name, alert_key,
    subject, count(*) AS notifications FROM notifications
    GROUP BY provider, account_uid, cluster_name, alert_key, subject
    HAVING count(*) > 1 ORDER BY notifications DESC""")

## Repair and rerun

Edit `pipeline.sql` in JupyterLab. Follow steps 2–4 in the lab guide. Rerun the cell below for a fresh attempt after each edit. A new connection to the same database resumes existing progress; it does not erase prior output.

In [ ]:
replay.close()
attempt.cleanup()
attempt = tempfile.TemporaryDirectory(prefix='scheduled_detection-notebook-')
database = Path(attempt.name) / 'state.duckdb'
replay = Replay(database, schedule, pipeline)
report = replay.run()
report

## Restart and recovery

Use the CLI crash commands in the lab guide with fresh database paths to test both sides of a commit. After implementing recovery, run the next cell twice: the second backfill must add no notifications.

In [ ]:
report = replay.recover()
report

## Submit

Save `pipeline.sql` and an incident note: explain a missed sequence, a duplicate notification, your lateness policy, recovery behavior and visibility limits. Practice targets: 27 initial notifications, 3 historical coverage warnings, 3 canonical late events, 30 notifications after recovery. Verify keys, evidence and timing as well as counts. The organizer uses a different replay schedule.

When finished, close the connection and remove only this attempt’s temporary database.

In [ ]:
replay.close()
attempt.cleanup()